# Peptides-struct — Graph Diameter & Hasse Diameter Analysis

Two structural-complexity metrics per molecule, both **diameters** (longest shortest path), computed on 4 different graphs (test split only, 2,331 molecules):

- **atom** — plain atom-bond graph diameter (`graph_diameter`, from `peptides_struct_test_per_molecule.npz`)
- **ct6** — CT's 6-relation cell complex, **atom-restricted** Hasse diameter: longest shortest path between two *atoms*, but bond/ring cells can be used as shortcuts en route (`atom_hasse_diameter`, from `peptides_struct_hasse_per_molecule.npz`)
- **cin** — CIN's message-passing graph, same atom-restricted construction (`cin_atom_hasse_diameter`, from `peptides_struct_hasse_cin_per_molecule.npz`)
- **cinpp** — CIN++'s message-passing graph, same atom-restricted construction (`cinpp_atom_hasse_diameter`)

Being atom-restricted, all three (`ct6`/`cin`/`cinpp`) are **guaranteed ≤ `atom`'s graph_diameter** (same endpoints, strictly more edges available to route through) — a ratio ≤ 1.0 always.

GCN/GAT/GIN/CIN/CINpp use hp-tuned results. CT does **not** have a hp-tuned run yet, so it uses its latest non-tuned result instead (`results_struct_ct_{original,simple,full}_3.json`); no SchNet result exists at all. See `analyze_results_hptuned.ipynb` for details on why CT's `_3` suffix is the valid one to use. `index` is the plain 0..N-1 position in the PyG test split for every file used, so no realignment/lookup is needed.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

plt.rcParams['figure.dpi'] = 130
pd.set_option('display.float_format', '{:.4f}'.format)

RESULTS_DIR = Path('results')

In [ ]:
FILES = {
    'CT-original': 'results_struct_ct_original_3.json',
    'CT-simple':   'results_struct_ct_simple_3.json',
    'CT-full':     'results_struct_ct_full_3.json',
    'GCN':   'results_struct_gcn_hptuned.json',
    'GAT':   'results_struct_gat_hptuned.json',
    'GIN':   'results_struct_gin_hptuned.json',
    'CIN':   'results_struct_cin_hptuned.json',
    'CINpp': 'results_struct_cinpp_hptuned.json',
}
# CT files are NOT hp-tuned (no tuned run submitted yet) -- these are the latest
# non-tuned reruns (suffix _3), included as the best available CT numbers.

data = {}
for label, fname in FILES.items():
    p = RESULTS_DIR / fname
    if p.exists():
        with open(p) as f:
            data[label] = json.load(f)
    else:
        print(f'Missing: {fname}')

print(f'Loaded {len(data)} result files: {list(data.keys())}')

In [ ]:
COLORS = {
    'CT':    '#4C72B0',
    'GCN':   '#DD8452',
    'GAT':   '#55A868',
    'GIN':   '#C44E52',
    'CINpp': '#937860',
    'CIN':   '#8172B2',
}

def model_color(label):
    for prefix, c in COLORS.items():
        if label.startswith(prefix):
            return c
    return 'gray'

model_labels = list(data.keys())
n_models = len(model_labels)
MODEL_COLOR = {lbl: model_color(lbl) for lbl in model_labels}

def per_molecule_err(d):
    """index -> mean absolute error across the 11 targets and 3 seeded runs."""
    err_by_idx = {}
    for run in d['runs']:
        for p in run['predictions']:
            err = float(np.mean(np.abs(np.array(p['pred']) - np.array(p['true']))))
            err_by_idx.setdefault(p['index'], []).append(err)
    return {k: float(np.mean(v)) for k, v in err_by_idx.items()}

err_maps = {lbl: per_molecule_err(data[lbl]) for lbl in model_labels}
for lbl in model_labels:
    print(f'{lbl:10s} n_molecules={len(err_maps[lbl])}')

In [ ]:
PREFIXES = ('atom', 'ct6', 'cin', 'cinpp')

struct_npz = np.load('peptides_struct_test_per_molecule.npz')
assert (struct_npz['index'] == np.arange(len(struct_npz['index']))).all()
graph_diameter = struct_npz['graph_diameter'].astype(float)

hasse_npz = np.load('peptides_struct_hasse_per_molecule.npz')
assert (hasse_npz['index'] == np.arange(len(hasse_npz['index']))).all()
ct6_diameter = hasse_npz['atom_hasse_diameter'].astype(float)

cin_npz = np.load('peptides_struct_hasse_cin_per_molecule.npz')
assert (cin_npz['index'] == np.arange(len(cin_npz['index']))).all()
cin_diameter = cin_npz['cin_atom_hasse_diameter'].astype(float)
cinpp_diameter = cin_npz['cinpp_atom_hasse_diameter'].astype(float)

ARR = {'atom': graph_diameter, 'ct6': ct6_diameter, 'cin': cin_diameter, 'cinpp': cinpp_diameter}

def get_diameter(prefix, idx):
    return float(ARR[prefix][idx])

for p in PREFIXES:
    arr = ARR[p]
    print(f'{p:6s}  n={len(arr):,}  min={arr.min():.0f}  max={arr.max():.0f}  mean={arr.mean():.2f}')

## Overview of Statistics

Each molecule contributes one diameter value per graph, across all 2,331 test molecules.

In [ ]:
rows = []
for p in PREFIXES:
    arr = ARR[p]
    rows.append({
        'graph': p, 'dataset_mean': arr.mean(), 'dataset_std': arr.std(),
        'dataset_min': arr.min(), 'dataset_max': arr.max(), 'n_valid': len(arr),
    })

overview_df = pd.DataFrame(rows).set_index('graph')
display(overview_df.style.format(precision=2).background_gradient(subset=['dataset_mean'], cmap='RdYlGn_r'))

### Diameter distribution, by graph

In [ ]:
GRAPH_COLOR = {'atom': '#4C72B0', 'ct6': '#DD8452', 'cin': '#55A868', 'cinpp': '#C44E52'}

fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
for col, p in enumerate(PREFIXES):
    ax = axes[col]
    ax.hist(ARR[p], bins=40, color=GRAPH_COLOR[p], alpha=0.8)
    ax.set_title(p, fontsize=11)
    ax.set_xlabel('diameter', fontsize=9)
    if col == 0:
        ax.set_ylabel('count', fontsize=9)

plt.suptitle('Peptides-struct — per-molecule diameter distributions', fontsize=13, y=1.03)
plt.tight_layout()
plt.show()

## Diameter vs. Model Error — Every Model, Per Graph Type

For each of the 4 graphs, one row of scatter plots — every model shown side by side, sharing the same x-axis (that graph's diameter for the molecule) against that model's own per-molecule mean absolute error (across the 11 targets). Dashed line = linear fit.

In [ ]:
def plot_err_vs_diameter(prefix):
    diam_arr = ARR[prefix]
    fig, axes = plt.subplots(1, n_models, figsize=(3.0 * n_models, 3.2), squeeze=False)
    axes = axes[0]
    for col, lbl in enumerate(model_labels):
        ax = axes[col]
        color = MODEL_COLOR[lbl]
        err_by_idx = err_maps[lbl]
        idxs = np.array(sorted(err_by_idx.keys()))
        x = diam_arr[idxs]
        y = np.array([err_by_idx[i] for i in idxs])
        ax.scatter(x, y, color=color, alpha=0.3, s=8, rasterized=True)
        if len(x) >= 2 and np.ptp(x) > 0:
            coeffs = np.polyfit(x, y, 1)
            x_line = np.linspace(x.min(), x.max(), 200)
            ax.plot(x_line, np.polyval(coeffs, x_line), color='black', linewidth=1.5,
                    linestyle='--', label=f'slope={coeffs[0]:.2e}')
            ax.legend(fontsize=6, loc='upper left')
        ax.set_title(lbl, fontsize=9)
        if col == 0:
            ax.set_ylabel('Absolute error', fontsize=9)
        ax.set_xlabel(f'{prefix} diameter', fontsize=8)
    plt.suptitle(f'Peptides-struct — error vs {prefix} diameter (every model)', fontsize=13, y=1.05)
    plt.tight_layout()
    plt.show()

plot_err_vs_diameter('atom')

In [ ]:
plot_err_vs_diameter('ct6')

In [ ]:
plot_err_vs_diameter('cin')

In [ ]:
plot_err_vs_diameter('cinpp')

## GNN vs. Other Architectures — Diameter by Outcome

For every non-GNN model (CT-original/simple/full, CIN, CINpp), split test molecules into two sets: **gnn_beats** (at least one of GCN/GAT/GIN has strictly lower error than this model on that molecule) vs. **model_beats** (this model's error was ≤ every GNN's there — it held its own). Same construction as `analyze_commute_time.ipynb`'s GNN-comparison sections (identical `err_maps`, so identical sets).

Four panels per model: **plain graph diameter** (atom) by outcome, **this model's own** Hasse diameter (`ct6` for CT variants, `cin` for CIN, `cinpp` for CINpp) by outcome, the **difference** `hasse - atom` diameter by outcome (always ≤ 0, since Hasse is atom-restricted), and the **ratio** `hasse/atom` diameter by outcome. Same 3-color scheme: gray = overlap, red = `gnn_beats`-only excess, blue = `model_beats`-only excess (density-normalized histograms, stacked).

In [ ]:
GNN_MODELS = [lbl for lbl in model_labels if lbl in ('GCN', 'GAT', 'GIN')]
NON_GNN_MODELS = [lbl for lbl in model_labels if lbl not in GNN_MODELS]

def model_prefix(label):
    """Which graph type this model's own diameter comes from."""
    if label.startswith('CT'):
        return 'ct6'
    if label.startswith('CINpp'):
        return 'cinpp'
    if label.startswith('CIN'):
        return 'cin'
    raise ValueError(f'no prefix mapping for {label}')

def build_gnn_beats_set(model_label):
    """molecule indices present in model_label's AND every GNN's err_maps.
    gnn_beats = indices where the best GNN's error is strictly lower than
    model_label's error there; model_beats = the rest of the common set
    (model_label's error was <= every GNN's, i.e. it held its own)."""
    common = set(err_maps[model_label])
    for g in GNN_MODELS:
        common &= set(err_maps[g])
    beats = {k for k in common
             if min(err_maps[g][k] for g in GNN_MODELS) < err_maps[model_label][k]}
    return beats, common - beats

gnn_beats_sets = {m: build_gnn_beats_set(m) for m in NON_GNN_MODELS}
for m in NON_GNN_MODELS:
    gnn_beats, model_beats = gnn_beats_sets[m]
    print(f'{m:12s}  gnn_beats={len(gnn_beats)}  model_beats={len(model_beats)}')

In [ ]:
OUTCOME_COLOR = {'gnn_beats': '#C44E52', 'model_beats': '#4C72B0'}
OVERLAP_COLOR = '#999999'

def _diam_vals(prefix, keys):
    arr = ARR[prefix]
    return np.array([arr[k] for k in keys], dtype=float)

def _diff_vals(prefix, keys):
    num, den = ARR[prefix], ARR['atom']
    return np.array([num[k] - den[k] for k in keys], dtype=float)

def _ratio_vals(prefix, keys):
    num, den = ARR[prefix], ARR['atom']
    vals = []
    for k in keys:
        d = den[k]
        if d == 0:
            continue
        vals.append(num[k] / d)
    return np.array(vals, dtype=float)

def _stacked_hist(ax, vals_a, vals_b, label_a, label_b, color_a, color_b, bins=30, integer=False):
    """Overlap (gray) as base, per-bin excess of whichever group is taller
    stacked on top in that group's own color. Bars are translucent with a
    solid-color outline so overlapping regions stay readable. integer=True
    uses one bin per integer value so bars touch with no gaps except where a
    diameter value truly has zero count -- used for the 3 integer-valued
    panels (diameter has no fractional values); the ratio panel stays
    continuous (default 30 bins)."""
    all_vals = np.concatenate([vals_a, vals_b])
    if integer:
        lo, hi = int(np.floor(all_vals.min())), int(np.ceil(all_vals.max()))
        edges = np.arange(lo - 0.5, hi + 1.5, 1.0)
    else:
        edges = np.histogram_bin_edges(all_vals, bins=bins)
    h_a, _ = np.histogram(vals_a, bins=edges, density=True)
    h_b, _ = np.histogram(vals_b, bins=edges, density=True)
    overlap = np.minimum(h_a, h_b)
    excess_a = h_a - overlap
    excess_b = h_b - overlap
    width = edges[1] - edges[0]
    centers = (edges[:-1] + edges[1:]) / 2
    ax.bar(centers, overlap, width=width, color=OVERLAP_COLOR, alpha=0.55,
           edgecolor=OVERLAP_COLOR, linewidth=0.9, label='overlap')
    ax.bar(centers, excess_a, width=width, bottom=overlap, color=color_a, alpha=0.55,
           edgecolor=color_a, linewidth=0.9, label=f'{label_a} (n={len(vals_a)})')
    ax.bar(centers, excess_b, width=width, bottom=overlap, color=color_b, alpha=0.55,
           edgecolor=color_b, linewidth=0.9, label=f'{label_b} (n={len(vals_b)})')

def plot_diameter_hist_by_outcome(model_label):
    p = model_prefix(model_label)
    gnn_beats, model_beats = gnn_beats_sets[model_label]
    fig, axes = plt.subplots(1, 4, figsize=(17, 4))

    ax = axes[0]
    _stacked_hist(ax, _diam_vals('atom', gnn_beats), _diam_vals('atom', model_beats),
                  'gnn_beats', 'model_beats', OUTCOME_COLOR['gnn_beats'], OUTCOME_COLOR['model_beats'],
                  integer=True)
    ax.set_xlabel('atom graph diameter', fontsize=9)
    ax.set_ylabel('density', fontsize=9)
    ax.set_title('graph diameter', fontsize=10)
    ax.legend(fontsize=7)

    ax = axes[1]
    _stacked_hist(ax, _diam_vals(p, gnn_beats), _diam_vals(p, model_beats),
                  'gnn_beats', 'model_beats', OUTCOME_COLOR['gnn_beats'], OUTCOME_COLOR['model_beats'],
                  integer=True)
    ax.set_xlabel(f'{p} diameter', fontsize=9)
    ax.set_title(f'{p} Complex diameter', fontsize=10)

    ax = axes[2]
    _stacked_hist(ax, _diff_vals(p, gnn_beats), _diff_vals(p, model_beats),
                  'gnn_beats', 'model_beats', OUTCOME_COLOR['gnn_beats'], OUTCOME_COLOR['model_beats'],
                  integer=True)
    ax.set_xlabel(f'{p} - atom diameter', fontsize=9)
    ax.set_title('diameter difference', fontsize=10)

    ax = axes[3]
    _stacked_hist(ax, _ratio_vals(p, gnn_beats), _ratio_vals(p, model_beats),
                  'gnn_beats', 'model_beats', OUTCOME_COLOR['gnn_beats'], OUTCOME_COLOR['model_beats'])
    ax.set_xlabel(f'{p}/atom diameter ratio', fontsize=9)
    ax.set_title('ratio', fontsize=10)

    plt.suptitle(f'Peptides-struct — diameter by outcome: GNN vs {model_label} ({p})', fontsize=12, y=1.03)
    plt.tight_layout()
    plt.show()

plot_diameter_hist_by_outcome('CT-original')

In [ ]:
plot_diameter_hist_by_outcome('CT-simple')

In [ ]:
plot_diameter_hist_by_outcome('CT-full')

In [ ]:
plot_diameter_hist_by_outcome('CIN')

In [ ]:
plot_diameter_hist_by_outcome('CINpp')